In [1]:
# 1. 사전 환경 준비
# !pip install shap wandb optuna-integration python-dotenv

In [2]:
import os
import wandb
from dotenv import load_dotenv

# 2. W&B 로그인 (env 파일에 저장된 API 키 로드)
load_dotenv()
wandb_key = os.getenv("WANDB_API_KEY")
if wandb_key and wandb_key != "your_wandb_api_key_here":
    wandb.login(key=wandb_key)
    print("W&B Logged in successfully!")
else:
    print("W&B API key not found in .env. Please set it!")
    wandb.login()  # Interactive prompt fallback

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/admin/.netrc
wandb: Currently logged in as: genie0320 (genie0320-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B Logged in successfully!


In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from scipy.special import logit, expit

data_dir = "." if os.path.exists("train.csv") else "../.."
train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))
print("Train shape:", train.shape, "Test shape:", test.shape)

Train shape: (3000, 18) Test shape: (3000, 17)


In [4]:
# ==========================================
# 3. 결측치 처리 및 Data-Centric 보강
# ==========================================
for df in [train, test]:
    df["medical_history"] = df["medical_history"].fillna("none")
    df["family_medical_history"] = df["family_medical_history"].fillna("none")
    df["edu_level"] = df["edu_level"].fillna("Unknown")

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# IterativeImputer를 사용하여 수치형 변수 대치 (mean_working 제외)
numeric_cols_for_imputation = [
    "age",
    "height",
    "weight",
    "cholesterol",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "glucose",
    "bone_density",
]

imputer = IterativeImputer(random_state=42, max_iter=20)

# Data Leakage 방지를 위해 Train 데이터로만 Imputer를 학습시킴
train[numeric_cols_for_imputation] = imputer.fit_transform(
    train[numeric_cols_for_imputation]
)
test[numeric_cols_for_imputation] = imputer.transform(test[numeric_cols_for_imputation])

# mean_working 결측치는 -1로 대치하여 트리가 결측 자체를 인식하게 유도
train["mean_working"] = train["mean_working"].fillna(-1)
test["mean_working"] = test["mean_working"].fillna(-1)

print("Imputation complete.")

Imputation complete.


In [5]:
# ==========================================
# 4. 파생 변수 추가 (원본 변수는 절대 삭제하지 않음)
# ==========================================
for df in [train, test]:
    # ag가 누락했던 결측치 시그널 파생 변수 강제 추가
    df["is_working_missing"] = (df["mean_working"] == -1.0).astype(int)
    # disease_count 파생 변수 (희소성 보완)
    df["medical_disease_count"] = df["medical_history"].apply(
        lambda x: 0 if x == "none" else len(str(x).split(","))
    )
    df["family_disease_count"] = df["family_medical_history"].apply(
        lambda x: 0 if x == "none" else len(str(x).split(","))
    )

    # 비선형 생체 지표 도메인 파생 변수 추가
    df["bmi"] = df["weight"] / ((df["height"] / 100) ** 2)
    df["pulse_pressure"] = (
        df["systolic_blood_pressure"] - df["diastolic_blood_pressure"]
    )
    df["map"] = (df["systolic_blood_pressure"] + 2 * df["diastolic_blood_pressure"]) / 3
    df["glucose_cholesterol_ratio"] = df["glucose"] / (df["cholesterol"] + 1)
    df["overwork_and_poor_sleep"] = (
        (df["mean_working"] >= 12) & (df["sleep_pattern"] == "sleep difficulty")
    ).astype(int)
    df["vascular_bone_risk"] = (
        (df["bone_density"] <= -1.0) & (df["pulse_pressure"] > 80)
    ).astype(int)

# 복합 질환(medical_history) 동적 이진화 플래그 생성 (train 기준으로 추출)
diseases = set()
for col in ["medical_history", "family_medical_history"]:
    for val in train[col].dropna().unique():
        for d in val.split(","):
            diseases.add(d.strip())
diseases.discard("none")
diseases = sorted(list(diseases))

for col, prefix in [("medical_history", "med"), ("family_medical_history", "fam")]:
    for disease in diseases:
        feat_name = f'{prefix}_{disease.lower().replace(" ", "_")}'
        train[feat_name] = train[col].apply(
            lambda x: 1 if disease in [d.strip() for d in x.split(",")] else 0
        )
        test[feat_name] = test[col].apply(
            lambda x: 1 if disease in [d.strip() for d in x.split(",")] else 0
        )

# Ordinal Encoding 매핑
edu_map = {
    "Unknown": 0,
    "high school diploma": 1,
    "bachelors degree": 2,
    "graduate degree": 3,
}
activity_map = {"light": 1, "moderate": 2, "intense": 3}
for df in [train, test]:
    # ag가 누락했던 결측치 시그널 파생 변수 강제 추가
    df["is_working_missing"] = (df["mean_working"] == -1.0).astype(int)
    df["edu_level_encoded"] = df["edu_level"].map(edu_map)
    df["activity_encoded"] = df["activity"].map(activity_map)

# 문자열 데이터는 Label Encoding
categorical_cols = [
    "gender",
    "smoke_status",
    "sleep_pattern",
    "activity",
    "edu_level",
    "medical_history",
    "family_medical_history",
]
for col in categorical_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train[col], test[col]]).astype(str))
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

print("Feature Engineering Complete. Train shape:", train.shape)

Feature Engineering Complete. Train shape: (3000, 35)


In [6]:
x_train = train.drop(["ID", "stress_score"], axis=1)
y_train = train["stress_score"]
x_test = test.drop("ID", axis=1)

In [7]:
# ==========================================
# 5. SHAP 기반 노이즈 제어 (완벽한 노이즈만 제명 + Whitelist 보호)
# ==========================================
import shap
import lightgbm as lgb
import matplotlib.pyplot as plt

print("Calculating SHAP values for baseline model feature selection...")

# Baseline LightGBM 학습 (SHAP 기여도 계산용)
baseline_model = lgb.LGBMRegressor(
    objective="regression_l1", random_state=42, verbose=-1, n_jobs=-1
)
baseline_model.fit(x_train, y_train)

# SHAP TreeExplainer 기여도 계산
explainer = shap.TreeExplainer(baseline_model)
shap_values = explainer.shap_values(x_train)

if isinstance(shap_values, list):
    shap_avg = np.abs(shap_values[0]).mean(axis=0)
else:
    shap_avg = np.abs(shap_values).mean(axis=0)

shap_imp = pd.Series(shap_avg, index=x_train.columns).sort_values(ascending=False)
print("\n--- SHAP Feature Importances (Absolute Mean) ---")
for feat, val in shap_imp.items():
    print(f"{feat}: {val:.6f}")

plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, x_train, plot_type="bar", show=False)
plt.tight_layout()
output_dir = "result/v11"
os.makedirs(output_dir, exist_ok=True)
plt.savefig(os.path.join(output_dir, "shap_summary_plot_v11.png"))
plt.close()

# [FIX] Whitelist: 도메인 파생 변수는 SHAP 기여도와 무관하게 절대 드롭 금지
WHITELIST = [
    "is_working_missing",
    "overwork_and_poor_sleep",
    "vascular_bone_risk",
    "mean_working",
]

# 오직 SHAP 기여도가 정확히 0.0인 피처만 제거 (Whitelist 제외)
features_to_drop = [f for f in shap_imp[shap_imp == 0.0].index if f not in WHITELIST]
print(f"\n[Whitelist Protected]: {WHITELIST}")
print(f"Removing features with exactly 0.0 SHAP importance: {features_to_drop}")
x_train = x_train.drop(
    columns=[f for f in features_to_drop if f in x_train.columns], errors="ignore"
)
x_test = x_test.drop(
    columns=[f for f in features_to_drop if f in x_test.columns], errors="ignore"
)
print(f"Final feature count: {x_train.shape[1]}")

Calculating SHAP values for baseline model feature selection...

--- SHAP Feature Importances (Absolute Mean) ---
mean_working: 0.021534
cholesterol: 0.021037
height: 0.020218
diastolic_blood_pressure: 0.016971
weight: 0.016383
glucose: 0.015571
map: 0.015561
bmi: 0.013650
glucose_cholesterol_ratio: 0.013382
bone_density: 0.013006
systolic_blood_pressure: 0.008798
pulse_pressure: 0.008215
smoke_status: 0.007798
age: 0.006986
edu_level: 0.006209
activity: 0.004682
sleep_pattern: 0.004525
family_disease_count: 0.004394
edu_level_encoded: 0.004289
medical_disease_count: 0.002637
family_medical_history: 0.002288
fam_diabetes: 0.002101
med_high_blood_pressure: 0.001804
gender: 0.001672
med_heart_disease: 0.001637
medical_history: 0.001472
med_diabetes: 0.001360
fam_heart_disease: 0.000607
fam_high_blood_pressure: 0.000208
is_working_missing: 0.000000
vascular_bone_risk: 0.000000
overwork_and_poor_sleep: 0.000000
activity_encoded: 0.000000

[Whitelist Protected]: ['is_working_missing', 'over

In [8]:
# ==========================================
# 6. Optuna Tuning (Logit Space + reg:absoluteerror Fix)
# ==========================================
import optuna
from optuna_integration.wandb import WeightsAndBiasesCallback
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import warnings

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

epsilon = 1e-5


# ---- XGBoost Tuning ----
def tune_xgboost(x_train, y_train):
    # [FIX] Logit 변환은 항상 무한대 공간으로 찢어놓음
    # [FIX] 이 공간에서는 반드시 reg:absoluteerror(절대오차)를 사용해야 안정적
    y_train_logit = logit(np.clip(y_train, epsilon, 1 - epsilon))

    def objective(trial):
        params = {
            # [CRITICAL FIX] objective 명시: Logit 공간의 무한대 값을 squarederror로
            # 학습하면 오차가 기하급수적으로 폭발함 → absoluteerror가 필수
            "objective": "reg:absoluteerror",
            "random_state": 42,
            "verbosity": 0,
            "n_jobs": 1,
            "enable_categorical": True,
            "tree_method": "hist",
            "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.003, 0.05, log=True
            ),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "subsample": trial.suggest_float("subsample", 0.5, 0.95),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.95),
            "gamma": trial.suggest_float("gamma", 0.0, 2.0),
            "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
            "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        }

        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        mae_scores = []

        for train_idx, val_idx in kf.split(x_train):
            X_tr, y_tr = x_train.iloc[train_idx], y_train_logit.iloc[train_idx]
            X_val = x_train.iloc[val_idx]
            y_val_orig = y_train.iloc[val_idx]

            model = xgb.XGBRegressor(**params)
            model.fit(
                X_tr,
                y_tr,
                eval_set=[(X_val, y_train_logit.iloc[val_idx])],
                verbose=False,
            )

            preds = np.clip(expit(model.predict(X_val)), 0.0, 1.0)
            mae_scores.append(mean_absolute_error(y_val_orig, preds))

        return np.mean(mae_scores)

    wandb_kwargs = {
        "project": "stress_index_v11",
        "name": "xgb_v11_fixed",
        "reinit": True,
    }
    wandbc = WeightsAndBiasesCallback(metric_name="mae", wandb_kwargs=wandb_kwargs)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=100, callbacks=[wandbc])
    wandb.finish()
    return study.best_params


# ---- LightGBM Tuning ----
def tune_lightgbm(x_train, y_train):
    # LightGBM도 Logit 공간에서 regression_l1 (MAE) 학습
    y_train_logit = logit(np.clip(y_train, epsilon, 1 - epsilon))

    def objective(trial):
        params = {
            "objective": "regression_l1",
            "random_state": 42,
            "verbose": -1,
            "n_jobs": 1,
            "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.003, 0.05, log=True
            ),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 30),
            "subsample": trial.suggest_float("subsample", 0.5, 0.95),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.95),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }

        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        mae_scores = []

        for train_idx, val_idx in kf.split(x_train):
            X_tr, y_tr = x_train.iloc[train_idx], y_train_logit.iloc[train_idx]
            X_val = x_train.iloc[val_idx]
            y_val_orig = y_train.iloc[val_idx]

            model = lgb.LGBMRegressor(**params)
            model.fit(
                X_tr,
                y_tr,
                eval_set=[(X_val, y_train_logit.iloc[val_idx])],
                callbacks=[
                    lgb.early_stopping(50, verbose=False),
                    lgb.log_evaluation(-1),
                ],
            )

            preds = np.clip(expit(model.predict(X_val)), 0.0, 1.0)
            mae_scores.append(mean_absolute_error(y_val_orig, preds))

        return np.mean(mae_scores)

    wandb_kwargs = {
        "project": "stress_index_v11",
        "name": "lgb_v11_tuning",
        "reinit": True,
    }
    wandbc = WeightsAndBiasesCallback(metric_name="mae", wandb_kwargs=wandb_kwargs)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=60, callbacks=[wandbc])
    wandb.finish()
    return study.best_params


print("=== XGBoost Hyperparameter Tuning (100 trials) ===")
best_xgb = tune_xgboost(x_train, y_train)
print(f"Best XGBoost Params: {best_xgb}")

print("\n=== LightGBM Hyperparameter Tuning (60 trials) ===")
best_lgb = tune_lightgbm(x_train, y_train)
print(f"Best LightGBM Params: {best_lgb}")

=== XGBoost Hyperparameter Tuning (100 trials) ===


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


alpha,▁▁▁▂▁▁▁▁▁▁▁▁▁▃▁▄▃▃▂▂█▂▁▁▁▁▄▆▂▁▂▅▂▁▂▁▁▁▁▁
colsample_bytree,▄█▅█▂▁▂▂▃▃▃▂▄▃▄▅▄▆▄▆▄▄▄▃▃▄▄▅▄▅▅▄▃▃▂▁▁▁▁▂
gamma,▃▂█▂▅▇▆▅▆▆▄▄▃▃▂▃▃▂▂▂▃▄▃▂▃▂▂▄▃▄▃▁▂▂▂▁▁▂▂▁
lambda,▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
learning_rate,▃▆▇▁▁██▄▂▄▁▆▆▅▅▅▅▇▅▄▅▇▄▆▃▃▆▅▂▄▄▄▄▃▄▂▃▃▃▂
mae,▆▆▄█▇▅▅▃▆▂▃▃▁▁▁▁▂▃▂▁▃▂▂▂▂▂▂▁▂▂▁▁▁▁▂▁▁▃▁▂
max_depth,▆▁▃▂▅▇█▇▇▆██▆▇▆█▇▃▁▆█▇█▆▇▇▇█▇███████████
min_child_weight,█▂▅▇▅▇▁▄▁▂▂▁▁▂▁▂▁█▂▁▁▂▁▁▂▁▂▁▁▁▂▂▂▂▂▂▁▁▁▁
n_estimators,▂▅▁▆█▇█▆▆▃▄▆▇▇▇▇▇▇▂▇▆▇█▆▆▇▇▆█▇▇▇█▇▆▆▅▆▆▄
subsample,▁▆▆█▁▃▁▄▃▃▅▆▅▇█▇█▇▇▇▇▆▅▆▇▅▇▆▄▆▇▇▇▇▇▇▇▇▇█
alpha,0.25012


Best XGBoost Params: {'n_estimators': 925, 'learning_rate': 0.021513320117781315, 'max_depth': 12, 'min_child_weight': 2, 'subsample': 0.9040329752237183, 'colsample_bytree': 0.5268485930396285, 'gamma': 0.15517667659583756, 'alpha': 0.4614227901476343, 'lambda': 0.17385955511359658}

=== LightGBM Hyperparameter Tuning (60 trials) ===


colsample_bytree,▄▄██▅▂▂▁▁▁▂▁▃▃▁▂▂▂▂▂▃▂▂▃▃▄▆▅▆▄▂▇▄▄▅▃▃▃▃▄
learning_rate,▁▅█▃▁▂▁▂███▄▆▃▇█▆█▆▆▇▅▇▄▆▆▇▆▅▇▁▅▅▄▃▄▆▆▅▇
mae,▇▆▄▆█▆▆▃▂▃▄▄▃▃▄▃▃▃▃▃▂▃▄▂▂▃▄▂▂▃▅▂▃▄▂▂▁▂▂▁
max_depth,▅▂▄▁▁▂████▆▇▅█▇▅▇█▆▅▆▆▅▆▆▆▅▆▆▅▆▆▃▇▅▆▆▅▄▅
min_child_samples,▄█▅▂▃▂▅▆▆▆▇▅█▅▇▇▆▅█▄▃▄▄▄▃▃▁▂▃▄▂▅▂▁▂▂▂▁▂▂
n_estimators,▂▂▇▄▂▂▆▃▇▅▄▅▄▃█▄▅▆▁▃█▆▆▇▇▇▇▆▆▇▆▇▆▇█▆▇█▇█
num_leaves,▁▃▂▇▇█▅▃▄▄▅▄▂▄▄▃▄▅▃▁▃▃▂▃▃▂▁▂▂▁▃▃▂▂▃▃▄▅▅▆
reg_alpha,▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▄▂█▁▂▁▁▁▁▃
reg_lambda,▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
subsample,▇▇▃▆▇▁▄▃▅▅▃▄▆▃█▆▄▅▄▆▆▇▇▆▆▇█▆▇▆▇▆▆▆██▇▇▇█
colsample_bytree,0.67346


Best LightGBM Params: {'n_estimators': 1199, 'learning_rate': 0.044837264482304684, 'max_depth': 9, 'num_leaves': 101, 'min_child_samples': 8, 'subsample': 0.9329714972853567, 'colsample_bytree': 0.6162265632637164, 'reg_alpha': 0.43205933310058514, 'reg_lambda': 0.003763327470456182}


In [9]:
# ==========================================
# 7. Seed Averaging + Ensemble + Quantile Rounding
# ==========================================


def train_xgb_with_seeds(x_train, y_train, x_test, best_params, seeds=[42, 2026, 777]):
    y_train_logit = logit(np.clip(y_train, epsilon, 1 - epsilon))
    oof_preds = np.zeros(len(x_train))
    test_preds = np.zeros(len(x_test))
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(kf.split(x_train)):
        X_tr, y_tr = x_train.iloc[train_idx], y_train_logit.iloc[train_idx]
        X_val = x_train.iloc[val_idx]
        fold_val_preds = np.zeros(len(X_val))
        fold_test_preds = np.zeros(len(x_test))

        for seed in seeds:
            params = best_params.copy()
            params.update(
                {
                    "objective": "reg:absoluteerror",  # [CRITICAL FIX]
                    "random_state": seed,
                    "verbosity": 0,
                    "enable_categorical": True,
                    "tree_method": "hist",
                    "n_jobs": 1,
                }
            )

            model = xgb.XGBRegressor(**params)
            model.fit(
                X_tr,
                y_tr,
                eval_set=[(X_val, y_train_logit.iloc[val_idx])],
                verbose=False,
            )

            fold_val_preds += np.clip(expit(model.predict(X_val)), 0.0, 1.0) / len(
                seeds
            )
            fold_test_preds += np.clip(expit(model.predict(x_test)), 0.0, 1.0) / len(
                seeds
            )

        oof_preds[val_idx] = fold_val_preds
        test_preds += fold_test_preds / 5

    mae = mean_absolute_error(y_train, oof_preds)
    print(f"[XGBoost] Seed Averaged OOF MAE: {mae:.6f}")
    return oof_preds, test_preds


def train_lgb_with_seeds(x_train, y_train, x_test, best_params, seeds=[42, 2026, 777]):
    y_train_logit = logit(np.clip(y_train, epsilon, 1 - epsilon))
    oof_preds = np.zeros(len(x_train))
    test_preds = np.zeros(len(x_test))
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(kf.split(x_train)):
        X_tr, y_tr = x_train.iloc[train_idx], y_train_logit.iloc[train_idx]
        X_val = x_train.iloc[val_idx]
        fold_val_preds = np.zeros(len(X_val))
        fold_test_preds = np.zeros(len(x_test))

        for seed in seeds:
            params = best_params.copy()
            params.update(
                {
                    "objective": "regression_l1",
                    "random_state": seed,
                    "verbose": -1,
                    "n_jobs": 1,
                }
            )

            model = lgb.LGBMRegressor(**params)
            model.fit(
                X_tr,
                y_tr,
                eval_set=[(X_val, y_train_logit.iloc[val_idx])],
                callbacks=[
                    lgb.early_stopping(50, verbose=False),
                    lgb.log_evaluation(-1),
                ],
            )

            fold_val_preds += np.clip(expit(model.predict(X_val)), 0.0, 1.0) / len(
                seeds
            )
            fold_test_preds += np.clip(expit(model.predict(x_test)), 0.0, 1.0) / len(
                seeds
            )

        oof_preds[val_idx] = fold_val_preds
        test_preds += fold_test_preds / 5

    mae = mean_absolute_error(y_train, oof_preds)
    print(f"[LightGBM] Seed Averaged OOF MAE: {mae:.6f}")
    return oof_preds, test_preds


# ---- 앙상블 실행 ----
print("=== Training XGBoost (3 seeds x 5 folds) ===")
oof_xgb, test_xgb = train_xgb_with_seeds(x_train, y_train, x_test, best_xgb)

print("=== Training LightGBM (3 seeds x 5 folds) ===")
oof_lgb, test_lgb = train_lgb_with_seeds(x_train, y_train, x_test, best_lgb)

# ---- 가중 평균 앙상블 (OOF로 최적 가중치 탐색) ----
best_weight = 0.5
best_oof_mae = float("inf")
for w in np.arange(0.3, 0.8, 0.05):
    oof_blend = w * oof_xgb + (1 - w) * oof_lgb
    oof_blend = np.clip(oof_blend, 0.0, 1.0)
    mae = mean_absolute_error(y_train, oof_blend)
    if mae < best_oof_mae:
        best_oof_mae = mae
        best_weight = w

print(f"\nBest blend weight (XGBoost): {best_weight:.2f}")
print(f"Blended OOF MAE: {best_oof_mae:.6f}")

final_oof = np.clip(best_weight * oof_xgb + (1 - best_weight) * oof_lgb, 0.0, 1.0)
test_preds_raw = np.clip(
    best_weight * test_xgb + (1 - best_weight) * test_lgb, 0.0, 1.0
)

# ---- [BREAKTHROUGH] Quantile Rounding ----
# 타겟은 0.01 단위(101개 클래스)로 이산화되어 있음
# 예측값도 0.01 단위로 반올림하면 MAE가 소폭 개선될 수 있음
test_preds_rounded = np.round(test_preds_raw * 100) / 100
oof_rounded = np.round(final_oof * 100) / 100

mae_raw = mean_absolute_error(y_train, final_oof)
mae_rounded = mean_absolute_error(y_train, oof_rounded)
print(f"\nOOF MAE (raw):    {mae_raw:.6f}")
print(f"OOF MAE (rounded): {mae_rounded:.6f}")

# 반올림이 도움되는 경우만 채택
if mae_rounded < mae_raw:
    test_preds = test_preds_rounded
    print("→ Rounding HELPS: using rounded predictions")
else:
    test_preds = test_preds_raw
    print("→ Rounding does not help: using raw predictions")

print(f"\nFinal OOF MAE: {min(mae_raw, mae_rounded):.6f}")
print(f"Prediction std: {test_preds.std():.4f} (target std: {y_train.std():.4f})")

=== Training XGBoost (3 seeds x 5 folds) ===
[XGBoost] Seed Averaged OOF MAE: 0.167075
=== Training LightGBM (3 seeds x 5 folds) ===
[LightGBM] Seed Averaged OOF MAE: 0.182325

Best blend weight (XGBoost): 0.75
Blended OOF MAE: 0.170201

OOF MAE (raw):    0.170201
OOF MAE (rounded): 0.170210
→ Rounding does not help: using raw predictions

Final OOF MAE: 0.170201
Prediction std: 0.1512 (target std: 0.2883)


In [10]:
submission = pd.read_csv(os.path.join(data_dir, "sample_submission.csv"))
submission["stress_score"] = test_preds
output_dir = "result/v11"
os.makedirs(output_dir, exist_ok=True)
submission.to_csv(os.path.join(output_dir, "submit_11.csv"), index=False)
print("Submission saved to result/v11/submit_11.csv")
print(f"Prediction range: [{test_preds.min():.4f}, {test_preds.max():.4f}]")
print(f"Prediction mean: {test_preds.mean():.4f}, std: {test_preds.std():.4f}")

Submission saved to result/v11/submit_11.csv
Prediction range: [0.0965, 0.9361]
Prediction mean: 0.4885, std: 0.1512
